<a href="https://colab.research.google.com/github/frank-morales2020/MITDevOps/blob/master/deepseekr1_api_demo_june2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://platform.deepseek.com/usage

In [57]:
!pip3 install openai -q

In [58]:
from google.colab import userdata

import os
deepseek_api_key = userdata.get('DEEPSEEK_API_KEY')

* The deepseek-chat model points to DeepSeek-V3-0324. You can invoke it by specifying model='deepseek-chat'.

* The deepseek-reasoner model points to DeepSeek-R1-0528. You can invoke it by specifying model='deepseek-reasoner'

https://huggingface.co/deepseek-ai/DeepSeek-R1-0528?utm_campaign=The%20Batch&utm_medium=email&_hsenc=p2ANqtz--M3NQ_S2i2a4-YigM_g7yAl65J52ZDkfTOAs4rKcQS5b4sBcaw8xHQxWdvtC9-m5_DNtdk92DS5nC5r5inVdTsTJvNZQ&_hsmi=364910056&utm_content=364905475&utm_source=hs_email

In [ ]:
# Please install OpenAI SDK first: `pip3 install openai`

from openai import OpenAI

client = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    #model="deepseek-chat",
    model="deepseek-reasoner",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Hello"},
    ],
    stream=False
)

print(response.choices[0].message.content)

Hello! 😊 How can I help you today?


In [ ]:
def deepseek_chat(prompt,model):
  response = client.chat.completions.create(
    #model="deepseek-chat",
    #model="deepseek-reasoner",
    model=model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": prompt},
    ],
    stream=False
  )

  print()
  print("-" * 80)
  print('Model: %s'%model)
  print("-" * 80)
  print('\n')

  print("-" * 80)
  print('Question: %s'%prompt)
  print("-" * 80)
  print('\n')

  print('Answer: ')
  print(response.choices[0].message.content)
  #return message.content[0].text
  return response.choices[0].message.content

In [ ]:
prompt = "What is the capital of France?"
response = deepseek_chat(prompt,'deepseek-chat')
print(response)


--------------------------------------------------------------------------------
Model: deepseek-chat
--------------------------------------------------------------------------------


--------------------------------------------------------------------------------
Question: What is the capital of France?
--------------------------------------------------------------------------------


Answer: 
The capital of France is **Paris**. 

Paris is known for its iconic landmarks such as the Eiffel Tower, the Louvre Museum, and the Notre-Dame Cathedral. It is also a major global hub for art, fashion, gastronomy, and culture. 

Let me know if you'd like more details!
The capital of France is **Paris**. 

Paris is known for its iconic landmarks such as the Eiffel Tower, the Louvre Museum, and the Notre-Dame Cathedral. It is also a major global hub for art, fashion, gastronomy, and culture. 

Let me know if you'd like more details!


In [ ]:
prompt= 'How do you plan out your trip? \
Bob is travelling to SAT from YVR \
1. He has a connection in DFW \
2. His connection is 6 hours long \
3. He has a budget of 100.00 including meals \
4. What can he do? Please suggest a time. \
5. Know- he is a hiker, museum, foodie, has a carry-on bag'

response = deepseek_chat(prompt,"deepseek-reasoner")
print(response)


--------------------------------------------------------------------------------
Model: deepseek-reasoner
--------------------------------------------------------------------------------


--------------------------------------------------------------------------------
Question: How do you plan out your trip? Bob is travelling to SAT from YVR 1. He has a connection in DFW 2. His connection is 6 hours long 3. He has a budget of 100.00 including meals 4. What can he do? Please suggest a time. 5. Know- he is a hiker, museum, foodie, has a carry-on bag
--------------------------------------------------------------------------------


Answer: 
Based on Bob's interests (hiking, museums, food), budget ($100), and constraints (6-hour layover at DFW with carry-on only), here's a tailored plan:

### **Recommended Itinerary:**
**Total Time Needed:** 4–5 hours (allowing 1–1.5 hours for security re-entry)  
**Budget Allocation:** $45–$65 (covers transit, activities, and a meal)  
**Focus:** Quick 

## TTP AGENT

Cell 1: Install Necessary Libraries

In [59]:
!pip install crewai -q
!pip install 'crewai[tools]' -q
!pip install openai -q

In [61]:
max_rpm=100
max_iterations=25
max_recursion_depth=10

Cell 2: Set Up DeepSeek API Key

In [62]:
from google.colab import userdata
import os

# Retrieve the DeepSeek API Key from Colab's userdata secrets
deepseek_api_key = userdata.get('DEEPSEEK_API_KEY')

if not deepseek_api_key:
    raise ValueError("DEEPSEEK_API_KEY not found in Colab secrets. Please add it.")

DEEPSEEK_API_KEY = deepseek_api_key # Assign to the variable used in subsequent cells

API definition

In [63]:
# Assuming you have a web search tool and a weather API tool defined similarly to DeepSeekChatTool
# For example (conceptual, you'd integrate actual API calls):
class WebSearchTool(BaseTool):
    name: str = "WebSearchTool"
    description: str = "A tool for performing general web searches to find information."

    def _run(self, query: str) -> str:
        # Placeholder for actual web search API call
        print(f"Performing web search for: {query}")
        return f"Search results for '{query}' (simulated)."

class WeatherAPITool(BaseTool):
    name: str = "WeatherAPITool"
    description: str = "A tool for retrieving current and forecasted weather conditions."

    def _run(self, location: str) -> str:
        # Placeholder for actual weather API call
        print(f"Retrieving weather for: {location}")
        return f"Current weather in {location}: Sunny, 25°C (simulated)."

web_search_tool = WebSearchTool()
weather_api_tool = WeatherAPITool()

Cell 3: Define a Custom Tool for DeepSeek Interaction (using BaseTool)

In [64]:
from crewai.tools import BaseTool
from openai import OpenAI
from pydantic import Field

class DeepSeekChatTool(BaseTool):
    name: str = "DeepSeekChatTool"
    description: str = "A powerful AI model for generating responses based on prompts."

    client: OpenAI = Field(default=None, exclude=True)

    # Add model_name as a field to be accessible in _run
    model_name: str = Field(default='deepseek-reasoner')

    def __init__(self, api_key: str, model_name: str = 'deepseek-reasoner', tool_name: str = "DeepSeek Reasoner", tool_description: str = "A powerful AI model capable of complex reasoning and generating detailed plans, useful for creating travel itineraries and solving logistical problems.", **data):
        # Ensure tool_name and tool_description are passed to the superclass
        super().__init__(name=tool_name, description=tool_description, **data)

        # Initialize client and model_name
        self.client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")
        self.model_name = model_name # Store model_name

    def _run(self, prompt: str) -> str:
        """
        Interacts with the DeepSeek model to get a response based on the prompt.
        """

        #print(f"Debug: Tool received prompt of type: {type(prompt)}")
        #print(f"Debug: Tool received prompt value: {prompt}")

        if not isinstance(prompt, str):
             # If the prompt is not a string, try to find the relevant text
             # This is a heuristic and might need adjustment based on what's actually passed
             if isinstance(prompt, dict) and 'description' in prompt:
                 actual_prompt = prompt['description']
                 #print(f"Debug: Extracted prompt from dictionary: {actual_prompt}")
             else:
                 #print("Debug: Could not extract string prompt from input.")
                 # Re-raise the error or handle it appropriately
                 raise ValueError(f"Expected prompt to be a string, but received {type(prompt)}")
        else:
            actual_prompt = prompt


        response = self.client.chat.completions.create(
            model=self.model_name, # Use the stored model_name
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": actual_prompt}, # Use the potentially extracted prompt
            ],
            stream=False
        )
        return response.choices[0].message.content

# Instantiate the tool with model_name being explicitly passed
deepseek_reasoner_tool = DeepSeekChatTool(
    api_key=DEEPSEEK_API_KEY,
    model_name='deepseek-reasoner',
    tool_name="DeepSeek Reasoner",
    tool_description="A powerful AI model capable of complex reasoning and generating detailed plans, useful for creating travel itineraries and solving logistical problems."
)

deepseek_chat_tool = DeepSeekChatTool(
    api_key=DEEPSEEK_API_KEY,
    model_name='deepseek-chat',
    tool_name="DeepSeek Chat",
    tool_description="A general-purpose AI model for conversational tasks and information retrieval."
)

Cell 4: Define Your Agents

In [65]:
from crewai import Agent

# Agent 1: Itinerary Planner
itinerary_planner = Agent(
    role='Expert Travel Itinerary Creator',
    goal='Create comprehensive and personalized tourism travel itineraries based on user interests, budget, and time constraints.',
    backstory="You are a seasoned travel agent with a knack for crafting perfect trips. You use advanced AI to generate detailed plans.",
    tools=[deepseek_reasoner_tool], # This agent needs the reasoning capability
    verbose=False
)

# Agent 2: Local Activities Recommender
local_activities_recommender = Agent(
    role='Local Expert and Activity Specialist',
    goal='Suggest relevant local activities and attractions that align with traveler preferences and layover constraints.',
    backstory="You know all the hidden gems and popular spots in any city. You ensure the traveler makes the most of their time.",
    tools=[deepseek_reasoner_tool], # Can also use the reasoner for more nuanced suggestions
    verbose=False
)

# Agent 3: Budget and Logistics Manager
budget_logistics_manager = Agent(
    role='Financial and Logistical Advisor',
    goal='Manage the travel budget and provide practical logistical advice for transportation and timing.',
    backstory="You are meticulous with numbers and timings, ensuring the trip stays within budget and runs smoothly.",
    tools=[deepseek_reasoner_tool], # Reasoning is key for budget allocation and timing
    verbose=False
)

In [66]:
from crewai import Agent

# Existing agents (from your provided code)
# itinerary_planner = Agent(...)
# local_activities_recommender = Agent(...) # We will modify this to be more focused or replace it with the new local activities agent
# budget_logistics_manager = Agent(...)

# New Agent 4: Weather Forecaster
weather_forecaster = Agent(
    role='Expert Weather Forecaster',
    goal='Provide accurate and timely weather forecasts for all relevant locations in the itinerary.',
    backstory="You are a meteorologist specialized in providing concise and critical weather information for travelers, ensuring they are prepared for any conditions.",
    tools=[weather_api_tool, web_search_tool], # Uses a dedicated weather API and web search for additional context
    verbose=False
)

# New Agent 5: Transportation Coordinator
transportation_coordinator = Agent(
    role='Efficient Transportation Planner',
    goal='Plan and optimize all ground transportation segments of the trip, including airport transfers, local transit, and inter-city travel.',
    backstory="You are a logistics expert with deep knowledge of various transportation modes and services, always finding the most efficient and cost-effective routes.",
    tools=[web_search_tool, deepseek_reasoner_tool], # Uses web search for real-time data and reasoner for complex routing
    verbose=False
)

# New Agent 6: Local Experience Finder (more specific than the general local activities recommender)
local_experience_finder = Agent(
    role='Hyper-local Experience Curator',
    goal='Discover unique, authentic, and interest-aligned local activities and dining options, considering hidden gems and specific traveler preferences beyond general tourist spots.',
    backstory="You are a cultural enthusiast and local guide, always unearthing the most genuine experiences that perfectly match a traveler's niche interests, even on a tight schedule.",
    tools=[web_search_tool, deepseek_reasoner_tool], # Relies heavily on web search for fresh info and reasoner for tailored suggestions
    verbose=False
)

Cell 5: Define Your Tasks

In [67]:
from crewai import Task

# Task for the Itinerary Planner
plan_itinerary_task = Task(
    description=(
        """Generate a detailed tourism travel itinerary for a traveler with the following profile:\n"""
        """- Departing from YVR, connecting in DFW for 6 hours, final destination SAT.\n"""
        """- Budget: $100 including meals.\n"""
        """- Interests: hiker, museum lover, foodie.\n"""
        """- Has a carry-on bag only.\n"""
        """- Incorporate weather considerations for outdoor activities.\n""" # New
        """- Provide specific transportation recommendations for each segment.\n""" # New
    ),
    agent=itinerary_planner,
    output_file='dfw_layover_itinerary_gemini_2_0.md', # Changed filename for gemini 2.0
    expected_output="A markdown formatted, comprehensive tourism travel itinerary including timeline, budget breakdown, activities, key tips, weather insights, and transportation details."
)

# You might also create new tasks specifically for these agents, e.g.:
get_weather_forecast_task = Task(
    description="Retrieve the current and forecasted weather for DFW during the layover period.",
    agent=weather_forecaster,
    expected_output="The current and forecasted weather conditions for the DFW area during the specified layover duration." # Added expected_output
)

plan_local_transportation_task = Task(
    description="Identify the best transportation options (rideshare, public transit, walking) for getting between DFW airport and recommended local attractions, considering the 6-hour layover.",
    agent=transportation_coordinator,
    expected_output="A list of recommended transportation methods for getting around DFW during a 6-hour layover, including estimated costs and travel times." # Added expected_output
)

find_niche_local_activities_task = Task(
    description="Based on the traveler's interests (hiking, museums, foodie) and carry-on only constraint, suggest unique local experiences in the DFW area suitable for a 6-hour layover.",
    agent=local_experience_finder,
    expected_output="A list of unique, interest-aligned local activities and dining suggestions in DFW suitable for a short layover, considering the traveler's preferences and carry-on limit." # Added expected_output
)

Cell 6: Form the Crew and Run

In [68]:
from crewai import Crew, Process


crew = Crew(
    agents=[
        itinerary_planner,
        local_activities_recommender, # Keep if still useful, or remove if local_experience_finder replaces it
        budget_logistics_manager,
        weather_forecaster,       # New agent
        transportation_coordinator, # New agent
        local_experience_finder   # New agent
    ],
    tasks=[
        plan_itinerary_task,
        get_weather_forecast_task,      # New task
        plan_local_transportation_task, # New task
        find_niche_local_activities_task # New task
    ],
    verbose=False,
    process=Process.sequential # Agents execute tasks one after another
)

print("Starting the DFW Layover Tourism Planning Crew (VERSION 2.0 - enhanced with weather, transportation, and local activity agents)...")
result = crew.kickoff()

print("\n\n########################")
print("## DFW Layover Itinerary (IMA-TTP) ##")
print("########################\n")
print(result)

Starting the DFW Layover Tourism Planning Crew (VERSION 2.0 - enhanced with weather, transportation, and local activity agents)...
Retrieving weather for: Dallas/Fort Worth International Airport
Retrieving weather for: Dallas/Fort Worth
Retrieving weather for: Dallas Fort Worth
Performing web search for: best transportation options from DFW airport to local attractions 2023
Performing web search for: unique hiking experiences museums dining suggestions DFW area 6-hour layover


########################
## DFW Layover Itinerary (IMA-TTP) ##
########################

Here’s a curated list of unique DFW layover experiences combining hiking, museums, and food—all walkable from public transit, budget-friendly (<$30 excluding meals), and achievable within 4.5 hours (allowing 1.5 hours for airport transit/security):

### 1. **Grapevine Historic District (Via TEXRail)**  
*Best for: Compact charm + Texas heritage*  
- **Hiking**: Walk the **Grapevine Botanical Gardens** (free, 30 mins) with na

In [69]:
from crewai import Crew, Process

crew = Crew(
    agents=[itinerary_planner, local_activities_recommender, budget_logistics_manager],
    tasks=[plan_itinerary_task],
    verbose=False, # Increased verbosity for detailed logs
    process=Process.sequential # Agents execute tasks one after another
)

print("Starting the DFW Layover Tourism Planning Crew...")
result = crew.kickoff()
print("\n\n########################")
print("## DFW Layover Itinerary ##")
print("########################\n")
print(result)

Starting the DFW Layover Tourism Planning Crew...
 Maximum iterations reached. Requesting final answer.


########################
## DFW Layover Itinerary ##
########################

### Travel Itinerary for a Traveler from YVR to SAT (with a DFW Layover)

**Traveler Profile:**
- **Departure:** YVR (Vancouver International Airport)
- **Layover:** DFW (Dallas/Fort Worth International Airport) - 6 hours
- **Final Destination:** SAT (San Antonio International Airport)
- **Budget:** $100 (Including meals)
- **Interests:** Hiking, Museums, Food
- **Luggage:** Carry-on only

---

#### **Itinerary Overview**
1. **Departure from YVR:** Morning Flight (Check for exact flight times)
2. **Arrive at DFW:** Morning 
3. **Layover in DFW**: 6 hours
   - Activities at DFW
4. **Arrive at SAT:** Afternoon

---

### Detailed Itinerary

#### **1. Depart from YVR**
- **Flight Information:** 
  - Confirm flight details based on booking.
  
#### **2. Layover at DFW (6 Hours)**

**Time: 9:00 AM - 3:00 PM (H